In [ ]:
# Optional: nur falls in der Umgebung noch nicht vorhanden
!pip install --quiet jpype1 rdflib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.5/496.5 kB 20.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
from __future__ import annotations

import glob
import json
import os
from pathlib import Path
import shutil
import subprocess

import jpype
import jpype.imports
from rdflib import Graph

# ===== Konfiguration (Renku) =====
ifc_input = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/Models/FoC_Preserving_IFC-Gantenbein.ifc")
output_dir = Path("/home/renku/work/dcaonnextcloud-500gb/ICDD/WeingutGantenbein/Payload Documents/RDF Data")
base_uri = "http://dca.example.org/"

# Kandidaten fuer Java-Klassen (IFCtoLBD Build/Fork)
classpath_globs = [
    "./jars/*",
    "/home/renku/work/IFCtoLBD/**/target/*.jar",
    "/home/renku/work/IFCtoLBD/**/target/dependency/*.jar",
]

# ===== Hilfsfunktionen =====
def find_libjvm() -> Path | None:
    candidates = []

    # 1) Direkte Suche unter JAVA_HOME
    java_home = os.environ.get("JAVA_HOME")
    if java_home:
        candidates.extend(glob.glob(f"{java_home}/**/libjvm.so", recursive=True))

    # 2) Ueber installierten java-Binary-Pfad
    java_bin = shutil.which("java")
    if java_bin:
        java_bin_path = Path(java_bin).resolve()
        guessed_home = java_bin_path.parent.parent
        candidates.extend(glob.glob(f"{guessed_home}/**/libjvm.so", recursive=True))

    # 3) Typische Linux-Pfade
    for root in ["/usr/lib/jvm", "/usr/lib64/jvm", "/opt/java", "/opt/jdk"]:
        if Path(root).exists():
            candidates.extend(glob.glob(f"{root}/**/libjvm.so", recursive=True))

    # Beste Heuristik: server/libjvm.so bevorzugen
    candidates = [Path(c) for c in dict.fromkeys(candidates)]  # dedupe, order preserved
    preferred = [c for c in candidates if "/server/libjvm.so" in str(c)]
    if preferred:
        return preferred[0]
    return candidates[0] if candidates else None

def find_classpath_entries(patterns: list[str]) -> list[str]:
    entries: list[str] = []
    for pattern in patterns:
        for p in glob.glob(pattern, recursive=True):
            if p.endswith(".jar") and Path(p).is_file():
                entries.append(str(Path(p).resolve()))
    # dedupe
    entries = list(dict.fromkeys(entries))
    return entries

# ===== Vorbedingungen =====
if not ifc_input.exists():
    raise FileNotFoundError(f"IFC-Datei nicht gefunden: {ifc_input}")

java_bin = shutil.which("java")
print(f"java im PATH: {java_bin}")
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME', '(nicht gesetzt)')}")

if java_bin is not None:
    try:
        version_out = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT, text=True)
        print("\nJava-Version:\n" + version_out)
    except Exception as exc:
        print(f"Konnte java -version nicht lesen: {exc}")

libjvm_path = find_libjvm()
print(f"Gefundene libjvm.so: {libjvm_path}")

if libjvm_path is None:
    raise EnvironmentError(
        "JVMNotFoundException Analyse: Keine libjvm.so gefunden.\n"
        "Bitte in Renku ein JDK installieren und JAVA_HOME setzen, z. B.:\n"
        "- apt: sudo apt-get update && sudo apt-get install -y openjdk-17-jdk\n"
        "- danach: export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))\n"
        "Oder im Projekt-Environment Java als Dependency aufnehmen."
    )

# JAVA_HOME best effort setzen, falls leer
if not os.environ.get("JAVA_HOME"):
    guessed_home = libjvm_path.parent.parent.parent
    os.environ["JAVA_HOME"] = str(guessed_home)
    print(f"JAVA_HOME automatisch gesetzt auf: {os.environ['JAVA_HOME']}")

classpath_entries = find_classpath_entries(classpath_globs)
print(f"Gefundene JARs im Classpath: {len(classpath_entries)}")
if classpath_entries:
    print("Beispiel JAR:", classpath_entries[0])

if not classpath_entries:
    raise FileNotFoundError(
        "Keine IFCtoLBD-JARs gefunden.\n"
        "Pruefe den Fork-Build unter /home/renku/work/IFCtoLBD und passe classpath_globs an."
    )

# ===== JVM starten =====
if not jpype.isJVMStarted():
    jpype.startJVM(
        str(libjvm_path),
        "-Xms1g",
        "-Xmx6g",
        classpath=classpath_entries,
    )
    print("JVM erfolgreich gestartet.")
else:
    print("JVM laeuft bereits.")

# ===== IFCtoLBD via JPype =====
IFCtoLBDConverter = jpype.JClass("org.linkedbuildingdata.ifc2lbd.IFCtoLBDConverter")
ConversionProperties = jpype.JClass("org.linkedbuildingdata.ifc2lbd.ConversionProperties")

props = ConversionProperties()
props.setHasGeometry(False)
props.setHasBuildingProperties(False)
props.setHasBuildingElements(True)
props.setHasUnits(False)
props.setHasNonLBDElement(False)
props.setExportIfcOWL(False)

lbdconverter = IFCtoLBDConverter(base_uri, 1)
lbdconverter.convert(str(ifc_input), props)

lbd_jsonld = str(lbdconverter.getJSONLD())
g = Graph()
g.parse(data=json.loads(lbd_jsonld), format="json-ld")

output_dir.mkdir(parents=True, exist_ok=True)
output_ttl = output_dir / "FoC_Preserving_IFC-Gantenbein_python_level1.ttl"
g.serialize(destination=output_ttl, format="turtle")

print(f"Konvertierung erfolgreich. Triples: {len(g):,}")
print(f"Ausgabe gespeichert: {output_ttl}")

JVMNotFoundException: No JVM shared library file (libjvm.so) found. Try setting up the JAVA_HOME environment variable properly.